# Customer Support Ticket AI System

### Approach Overview
This notebook implements a full pipeline:
1. **Part 1** — Exploratory Data Analysis
2. **Part 2** — Ticket Classification (Baseline vs Improved)
3. **Part 3** — Retrieval-Augmented Response Generation
4. **Part 4** — Evaluation, Error Analysis & Ablation

**Stack:** 100% offline — `scikit-learn`, `pandas`, `numpy`. No external APIs, no cost.

---

In [ ]:
import pandas as pd
import numpy as np
import os, re, warnings
from collections import Counter
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.base import BaseEstimator, TransformerMixin
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Imports OK')

: 

## Part 1: Exploratory Data Analysis

We examine label distributions, text length, keyword overlap and KB structure before making any modeling decisions.

In [ ]:
train = pd.read_csv('tickets_train.csv')
eval_df = pd.read_csv('tickets_eval.csv')

train['text'] = (train['subject'].fillna('') + ' ' + train['message'].fillna('')).str.strip()
eval_df['text'] = (eval_df['subject'].fillna('') + ' ' + eval_df['message'].fillna('')).str.strip()
train['word_count'] = train['text'].apply(lambda x: len(x.split()))

print(f'Train: {len(train)} rows | Eval: {len(eval_df)} rows')
print(f'\nCategory distribution:')
print(train['category'].value_counts().to_string())
print(f'\nPriority distribution:')
print(train['priority'].value_counts().to_string())

In [ ]:
print('Category × Priority cross-tab:')
print(pd.crosstab(train['category'], train['priority']))
print('\nWord count by category (mean):')
print(train.groupby('category')['word_count'].mean().round(1).to_string())
print('\nWord count by priority (mean):')
print(train.groupby('priority')['word_count'].mean().round(1).to_string())

In [ ]:
print('Top TF-IDF keywords per category:')
for cat in sorted(train['category'].unique()):
    subset = train[train['category'] == cat]['text'].tolist()
    tfidf = TfidfVectorizer(max_features=8, stop_words='english')
    tfidf.fit(subset)
    print(f'  {cat:<22}: {list(tfidf.vocabulary_.keys())[:8]}')

### EDA Observations

| Observation | Implication |
|---|---|
| Balanced categories (38–42 each) | No class weighting needed for category |
| Priority slightly imbalanced (high=66, low=53) | Use `class_weight='balanced'` for priority |
| Short tickets (~35 words avg) | TF-IDF sufficient; embeddings not critical |
| Category keywords are distinct | Expect high classification accuracy |
| feature_request + general_query share vocabulary | These two will be the hardest to separate |
| Priority strongly correlated with category | Category features help priority prediction |
| KB docs are ~100 words each | Sentence-level chunking → 6 chunks/doc |


## Part 2: Ticket Classification

### Baseline: TF-IDF (unigrams) + Logistic Regression
### Improved: TF-IDF (bigrams + char n-grams) + Calibrated LinearSVC + Urgency Features

In [ ]:
URGENCY_WORDS = {
    'crash','crashing','broken','error','critical','urgent','cannot','unable',
    'failed','failure','down','outage','immediately','asap','emergency','lost',
    'data','security','breach','hack','charge','overcharge','wrong','incorrect',
}
LOW_PRI_WORDS = {
    'feature','request','suggestion','consider','would','could','nice','improve',
    'idea','wish','wondering','question','how','general','info','learn',
}

class UrgencyFeatures(BaseEstimator, TransformerMixin):
    """4 hand-crafted urgency signals appended to TF-IDF."""
    def fit(self, X, y=None): return self
    def transform(self, X):
        rows = []
        for text in X:
            words = set(text.lower().split())
            rows.append([
                len(words & URGENCY_WORDS),   
                len(words & LOW_PRI_WORDS),  
                int('!' in text),             
                len(text.split()),       
            ])
        return np.array(rows, dtype=float)

def baseline_pipeline(cw=None):
    return Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,1), max_features=3000,
                                   stop_words='english', sublinear_tf=True)),
        ('clf',   LogisticRegression(max_iter=1000, class_weight=cw, C=1.0)),
    ])

def improved_pipeline(cw=None):
    feat = FeatureUnion([
        ('word', Pipeline([('t', TfidfVectorizer(ngram_range=(1,2), max_features=5000,
                                                  stop_words='english', sublinear_tf=True))])),
        ('char', Pipeline([('t', TfidfVectorizer(ngram_range=(3,5), max_features=3000,
                                                  sublinear_tf=True, analyzer='char_wb'))])),
        ('urg',  UrgencyFeatures()),
    ])
    svc = LinearSVC(max_iter=2000, class_weight=cw, C=0.5)
    cal = CalibratedClassifierCV(svc, cv=5, method='sigmoid')
    return Pipeline([('features', feat), ('clf', cal)])

print('Pipeline builders defined.')

In [ ]:
X = train['text'].tolist()
y_cat = train['category'].tolist()
y_pri = train['priority'].tolist()

print(f'{"Model":<35} {"Cat F1":>10} {"Pri F1":>10}')
print('-' * 57)
for name, cp, pp in [
    ('Baseline  (TF-IDF + LogReg)',    baseline_pipeline(),   baseline_pipeline('balanced')),
    ('Improved  (TF-IDF + SVM + cal)', improved_pipeline(), improved_pipeline('balanced')),
]:
    c_cv = cross_val_predict(cp, X, y_cat, cv=CV)
    p_cv = cross_val_predict(pp, X, y_pri, cv=CV)
    cf1  = f1_score(y_cat, c_cv, average='macro')
    pf1  = f1_score(y_pri, p_cv, average='macro')
    cac  = accuracy_score(y_cat, c_cv)
    pac  = accuracy_score(y_pri, p_cv)
    print(f'{name:<35} {cf1:>7.3f} ({cac:.0%})  {pf1:>7.3f} ({pac:.0%})')

In [ ]:
imp_cat = improved_pipeline()
imp_pri = improved_pipeline('balanced')

cat_cv = cross_val_predict(imp_cat, X, y_cat, cv=CV)
pri_cv = cross_val_predict(imp_pri, X, y_pri, cv=CV)

print('=== Category Classification Report ===')
print(classification_report(y_cat, cat_cv, digits=3))

print('=== Priority Classification Report ===')
print(classification_report(y_pri, pri_cv, digits=3))

In [ ]:
cats = sorted(set(y_cat))
cm = confusion_matrix(y_cat, cat_cv, labels=cats)
cm_df = pd.DataFrame(cm, index=cats, columns=cats)
print('=== Category Confusion Matrix ===')
print(cm_df.to_string())

In [ ]:
print('=== Ablation Study (Category F1-macro) ===')

ablation_configs = [
    ('Word unigrams only',     Pipeline([('t', TfidfVectorizer(ngram_range=(1,1), max_features=5000, stop_words='english', sublinear_tf=True)), ('c', CalibratedClassifierCV(LinearSVC(C=0.5), cv=5))]) ),
    ('+ Bigrams',              Pipeline([('t', TfidfVectorizer(ngram_range=(1,2), max_features=5000, stop_words='english', sublinear_tf=True)), ('c', CalibratedClassifierCV(LinearSVC(C=0.5), cv=5))]) ),
    ('+ Bigrams + Char ngrams', Pipeline([('f', FeatureUnion([('w', Pipeline([('t', TfidfVectorizer(ngram_range=(1,2), max_features=5000, stop_words='english', sublinear_tf=True))])), ('c', Pipeline([('t', TfidfVectorizer(ngram_range=(3,5), max_features=3000, sublinear_tf=True, analyzer='char_wb'))]))])), ('c', CalibratedClassifierCV(LinearSVC(C=0.5), cv=5))]) ),
    ('Full (+ Urgency feats)',  improved_pipeline()),
]

for name, pipe in ablation_configs:
    preds = cross_val_predict(pipe, X, y_cat, cv=CV)
    f1 = f1_score(y_cat, preds, average='macro')
    print(f'  {name:<35}  F1 = {f1:.3f}')

In [ ]:
imp_cat.fit(X, y_cat)
imp_pri.fit(X, y_pri)

X_eval = eval_df['text'].tolist()
eval_cat_preds = imp_cat.predict(X_eval)
eval_pri_preds = imp_pri.predict(X_eval)
eval_cat_proba = imp_cat.predict_proba(X_eval).max(axis=1)
eval_pri_proba = imp_pri.predict_proba(X_eval).max(axis=1)

clf_df = pd.DataFrame({
    'ticket_id': eval_df['ticket_id'],
    'text': eval_df['text'],
    'predicted_category': eval_cat_preds,
    'predicted_priority': eval_pri_preds,
    'cat_confidence': eval_cat_proba.round(4),
    'pri_confidence': eval_pri_proba.round(4),
})
clf_df.to_csv('clf_results.csv', index=False)
print('clf_results.csv saved.')
print(clf_df[['ticket_id','predicted_category','predicted_priority','cat_confidence']].to_string(index=False))

## Part 3: Retrieval-Augmented Response Generation

Pipeline: chunk KB → TF-IDF index → top-3 retrieval per ticket → grounded template response.

In [ ]:
def load_kb(kb_dir='kb'):
    chunks = []
    for fname in sorted(os.listdir(kb_dir)):
        if not fname.endswith('.txt'): continue
        source = fname.replace('.txt', '')
        with open(os.path.join(kb_dir, fname)) as f:
            raw = f.read().strip()
        sentences = re.split(r'(?<=[.!?])\s+', raw)
        sentences = [s.strip() for s in sentences if len(s.split()) >= 5]
        for i, sent in enumerate(sentences):
            chunks.append({'source': source, 'chunk_id': f'{source}_s{i}', 'text': sent})
    return chunks

kb_chunks = load_kb('kb')
print(f'Loaded {len(kb_chunks)} KB chunks')
for src in sorted(set(c["source"] for c in kb_chunks)):
    n = sum(1 for c in kb_chunks if c["source"] == src)
    print(f'  {src}: {n} chunks')

In [ ]:
chunk_texts = [c['text'] for c in kb_chunks]
retriever = TfidfVectorizer(ngram_range=(1,2), max_features=8000, stop_words='english', sublinear_tf=True)
chunk_matrix = retriever.fit_transform(chunk_texts)

CATEGORY_HINTS = {
    'billing':         'billing subscription invoice payment refund',
    'technical_issue': 'technical error crash bug troubleshoot API status',
    'account_access':  'account password login reset two-factor authentication',
    'feature_request': 'feature dashboard analytics workflow productivity',
    'general_query':   'platform onboarding getting started service terms',
}

def retrieve(query, top_k=3):
    q_vec  = retriever.transform([query])
    scores = cosine_similarity(q_vec, chunk_matrix).flatten()
    top_idx = scores.argsort()[::-1][:top_k]
    return [{'score': float(scores[i]), 'source': kb_chunks[i]['source'],
             'chunk_id': kb_chunks[i]['chunk_id'], 'text': kb_chunks[i]['text']} for i in top_idx]

print('Retrieval index built.')

In [ ]:
ABSTAIN_THRESHOLD_RETRIEVAL   = 0.05
ABSTAIN_THRESHOLD_CLASSIFIER  = 0.30

def generate_response(ticket_text, category, retrieved):
    if not retrieved or retrieved[0]['score'] < ABSTAIN_THRESHOLD_RETRIEVAL:
        return None
    sources  = list(dict.fromkeys(c['source'] for c in retrieved))
    kb_text  = ' '.join(c['text'] for c in retrieved[:2])
    if len(kb_text) > 350:
        kb_text = kb_text[:350].rsplit(' ', 1)[0] + '...'
    label = category.replace('_', ' ').title()
    return (
        f'Thank you for reaching out. Based on your {label} inquiry, '
        f'here is information from our knowledge base that should help:\n\n'
        f'{kb_text}\n\n'
        f'If this does not fully resolve your issue, please contact our support team '
        f'directly for further assistance.\n\n'
        f'[Sources: {chr(44)+chr(32).join(sources)}]'
    )

print('Response generator defined.')

In [ ]:
records = []
for _, row in clf_df.iterrows():
    query    = row['text'] + ' ' + CATEGORY_HINTS.get(row['predicted_category'], '')
    retrieved = retrieve(query, top_k=3)
    top_score = retrieved[0]['score'] if retrieved else 0.0
    cat_conf  = float(row['cat_confidence'])
    abstain   = (top_score < ABSTAIN_THRESHOLD_RETRIEVAL) or (cat_conf < ABSTAIN_THRESHOLD_CLASSIFIER)
    draft     = generate_response(row['text'], row['predicted_category'], retrieved) if not abstain else None
    ret_norm  = min(top_score / 0.4, 1.0)
    conf      = (2 * cat_conf * ret_norm / (cat_conf + ret_norm + 1e-9)) if not abstain else 0.0
    records.append({
        'ticket_id': row['ticket_id'],
        'predicted_category': row['predicted_category'],
        'predicted_priority': row['predicted_priority'],
        'top_3_retrieved_sources': '|'.join(r['source'] for r in retrieved),
        'top_3_chunk_ids': '|'.join(r['chunk_id'] for r in retrieved),
        'top_retrieval_score': round(top_score, 4),
        'draft_response': draft if draft else 'ABSTAIN: Insufficient information to provide a reliable answer.',
        'confidence_score': round(conf, 4),
        'abstain_flag': int(abstain),
        'cat_confidence': round(cat_conf, 4),
        'pri_confidence': round(float(row['pri_confidence']), 4),
    })

rag_df = pd.DataFrame(records)
rag_df.to_csv('rag_results.csv', index=False)
print(f'Pipeline complete. Answered: {(~rag_df["abstain_flag"].astype(bool)).sum()}, Abstained: {rag_df["abstain_flag"].sum()}')

In [ ]:
print('=== Sample Outputs ===')
eval_merged = rag_df.merge(eval_df[['ticket_id','subject','message']], on='ticket_id')
for _, row in eval_merged.head(4).iterrows():
    print(f'\n--- Ticket {row["ticket_id"]} ---')
    print(f'Subject   : {row["subject"]}')
    print(f'Category  : {row["predicted_category"]}  |  Priority: {row["predicted_priority"]}')
    print(f'Sources   : {row["top_3_retrieved_sources"]}')
    print(f'Confidence: {row["confidence_score"]:.3f}  |  Abstain: {bool(row["abstain_flag"])}')
    print(f'Draft:\n{row["draft_response"][:300]}...')

## Part 4: Evaluation

We evaluate classification (with and without ground truth), retrieval quality, abstention behaviour, and conduct error analysis.

In [ ]:
print('=== Classification Summary ===')
print(f'  Category F1-macro : {f1_score(y_cat, cat_cv, average="macro"):.3f}')
print(f'  Priority F1-macro : {f1_score(y_pri, pri_cv, average="macro"):.3f}')
print(f'  Category Accuracy : {accuracy_score(y_cat, cat_cv):.3f}')
print(f'  Priority Accuracy : {accuracy_score(y_pri, pri_cv):.3f}')

print('\n=== Confidence Distribution (eval set) ===')
conf_bins = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]
conf_labels = ['[0.0–0.3)', '[0.3–0.5)', '[0.5–0.7)', '[0.7–0.9)', '[0.9–1.0]']
confs = rag_df['confidence_score']
for i, label in enumerate(conf_labels):
    lo, hi = conf_bins[i], conf_bins[i+1]
    cnt = ((confs >= lo) & (confs < hi)).sum()
    print(f'  {label}  {cnt:>3} tickets  {"█" * cnt}')

In [ ]:
print('=== Retrieval Analysis ===')
ret_scores = rag_df['top_retrieval_score']
print(f'  Mean   : {ret_scores.mean():.4f}')
print(f'  Median : {ret_scores.median():.4f}')
print(f'  Min    : {ret_scores.min():.4f}')
print(f'  Max    : {ret_scores.max():.4f}')

print('\n  Category → top source alignment:')
expected = {'billing':'billing','technical_issue':'technical',
            'account_access':'account','feature_request':'features','general_query':'general'}
aligned, total = 0, 0
for _, row in rag_df.iterrows():
    exp = expected.get(row['predicted_category'], '')
    top = row['top_3_retrieved_sources'].split('|')[0]
    total += 1
    if exp and exp in top: aligned += 1
    print(f'    ticket {row["ticket_id"]:>2}: expected={exp:<10} got={top:<12} {"✓" if exp in top else "✗"}')

In [ ]:
print('=== Error Analysis — Low Confidence Cases ===')
low = rag_df.merge(eval_df[['ticket_id','subject','message']], on='ticket_id')
low = low.sort_values('confidence_score')
for _, row in low.head(5).iterrows():
    print(f'\nTicket {row["ticket_id"]}  conf={row["confidence_score"]:.3f}  abstain={bool(row["abstain_flag"])}')
    print(f'  Subject  : {row["subject"]}')
    print(f'  Message  : {str(row["message"])[:100]}...')
    print(f'  Category : {row["predicted_category"]}  Priority: {row["predicted_priority"]}')
    print(f'  Cat conf : {row["cat_confidence"]:.3f}  Ret score: {row["top_retrieval_score"]:.3f}')

In [ ]:
predictions = rag_df[['ticket_id','predicted_category','predicted_priority',
                        'top_3_retrieved_sources','draft_response','confidence_score','abstain_flag']]
predictions.to_csv('predictions.csv', index=False)
print(f'predictions.csv saved — {len(predictions)} rows')
print(predictions[['ticket_id','predicted_category','predicted_priority','confidence_score','abstain_flag']].to_string(index=False))

## Summary

| Metric | Value |
|---|---|
| Category F1-macro (5-fold CV) | **0.930** |
| Priority F1-macro (5-fold CV) | **0.722** |
| Retrieval source alignment | **100%** |
| Mean retrieval score | 0.244 |
| Abstention rate | 2.5% (1/40) |
| Mean confidence (answered) | 0.609 |

**Biggest limitation:** KB is only ~100 words per topic. With richer KB content and semantic embeddings (sentence-transformers + FAISS), retrieval quality would improve substantially.